In [1]:
import pandas as pd
import csv
import re
import numpy as np
import pandas as pd
import sys
import re
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests
from neutrality_test import *
from bottleneck_function import *

#### Filter out the sgRNA suspected selection (down-nonneutral)

In [2]:
neutrality_df = pd.read_csv("essential_gene_invivo_notvitro.csv")
remove_set = set(neutrality_df.loc[0:266, "feature"].astype(str).str.strip())

with open("Table-S7-sgRNA-Raw-Counts.csv", "r", newline="", encoding="utf-8", errors="replace") as fin, \
     open("refined_S7_rawcount.csv", "w", newline="", encoding="utf-8") as fout:
    reader = csv.reader(fin)
    writer = csv.writer(fout)

    # Preserve original headers exactly (line 1 + line 2)
    writer.writerow(next(reader))  # original line 1
    writer.writerow(next(reader))  # original line 2 (column names)

    # Filter remaining lines by Geneid (first column)
    for row in reader:
        if row and row[0].strip() in remove_set:
            continue
        writer.writerow(row)

#### Compute the bottleneck again

In [3]:
def S(x) -> str:
    return str(x).strip()

def build_groups_from_merged_header(df_raw: pd.DataFrame):
    cols = list(df_raw.columns)
    groups = []
    cur = None
    for c in cols:
        if not str(c).startswith("Unnamed"):
            cur = c
        groups.append(cur)
    return groups

def find_col_by_sample_and_group(df_raw, sample_names, groups, sample_label, group_keyword):
    gkw = group_keyword.lower()
    idxs = [i for i, nm in enumerate(sample_names) if S(nm) == sample_label and (groups[i] and gkw in str(groups[i]).lower())]
    if not idxs:
        raise ValueError(f"Cannot find sample '{sample_label}' inside group containing '{group_keyword}'.")
    return df_raw.columns[idxs[0]]

def idxs_by_regex_and_group(sample_names, groups, pattern, group_keyword):
    pat = re.compile(pattern)
    gkw = group_keyword.lower()
    out = []
    for i, nm in enumerate(sample_names):
        if pat.fullmatch(S(nm)) and (groups[i] and gkw in str(groups[i]).lower()):
            out.append(i)
    return out

def get_counts_vector(sgrna_df: pd.DataFrame, colname) -> np.ndarray:
    return (pd.to_numeric(sgrna_df[colname], errors="coerce").fillna(0).round().astype(int).to_numpy())

In [4]:
df_raw = pd.read_csv("refined_S7_rawcount.csv") 
sample_names = df_raw.iloc[0].astype(str).tolist()   # the *second* CSV row (sample names)

# data rows (sgRNA001..), drop fully-empty trailing row if present
sgrna_df = df_raw.iloc[1:].copy()
sgrna_df = sgrna_df.dropna(how="all")

groups = build_groups_from_merged_header(df_raw)

# ---------------------------- 24 hpi ----------------------------
PRE24_GROUP = "preinfection samples for 24 hpi"
H24_GROUP   = "24 hpi of murine pneumonia model"

donor_pre1_24_col = find_col_by_sample_and_group(df_raw, sample_names, groups, "Pre1", PRE24_GROUP)
donor_pre2_24_col = find_col_by_sample_and_group(df_raw, sample_names, groups, "Pre2", PRE24_GROUP)

donor_pre1_24 = get_counts_vector(sgrna_df, donor_pre1_24_col)  # noDox donor (as in notebook)
donor_pre2_24 = get_counts_vector(sgrna_df, donor_pre2_24_col)  # +dox donor (as in notebook)

nodox_24_idxs = idxs_by_regex_and_group(sample_names, groups, r"Mice_noDox_\d+",   H24_GROUP)
dox_24_idxs   = idxs_by_regex_and_group(sample_names, groups, r"Mice_plusDox_\d+", H24_GROUP)

records24 = []
for i in nodox_24_idxs:
    nm = S(sample_names[i])
    mouse = int(nm.split("_")[-1])
    col = df_raw.columns[i]
    x = get_counts_vector(sgrna_df, col)
    nb_hat = bottleneck_from_two_timepoints(donor_pre1_24, x)
    records24.append(("noDox", mouse, "Lung", nb_hat, nm))

for i in dox_24_idxs:
    nm = S(sample_names[i])
    mouse = int(nm.split("_")[-1])
    col = df_raw.columns[i]
    x = get_counts_vector(sgrna_df, col)
    nb_hat = bottleneck_from_two_timepoints(donor_pre2_24, x)
    records24.append(("Dox", mouse, "Lung", nb_hat, nm))

res24_long = pd.DataFrame(records24, columns=["condition", "mouse", "tissue", "Nb_hat", "sample"])

export24 = (res24_long.pivot_table(index=["condition", "mouse"], columns="tissue", values="Nb_hat", aggfunc="first").reset_index().rename_axis(None, axis=1))
export24["hpi"] = 24
export24["label"] = export24.apply(lambda r: f"#{int(r['mouse'])}" if r["condition"] == "noDox" else f"#{int(r['mouse'])}+dox",axis=1)
if "Blood" not in export24.columns:
    export24["Blood"] = np.nan
export24 = export24.rename(columns={"Lung": "lung", "Blood": "blood"})[["hpi","condition","mouse","label","lung","blood"]]
export24.to_csv("Nb_filtered.csv", index=False)

#### Test neutrality of refined data

In [5]:
raw = pd.read_csv("refined_S7_rawcount.csv")
sample_names = raw.iloc[0]

def normalize_sgrna_id(x) -> str: #Convert IDs like 'sgRNA1', 'sgRNA001', 1 -> 'sgRNA0001'. If no digits are found, returns the original string."""
    s = str(x).strip()
    m = re.search(r"(\d+)", s)
    if not m:
        return s
    num = m.group(1)
    return f"sgRNA{num.zfill(4)}"

# --- normalized sgRNA IDs (length K) ---
sgRNAs_raw = raw.iloc[1:, 0].astype(str).to_numpy()
sgRNAs = np.array([normalize_sgrna_id(v) for v in sgRNAs_raw], dtype=object)

In [6]:
def counts_for_cols(cols):
    """Return a (K x len(cols)) DataFrame of integer counts with index=normalized sgRNAs."""
    mat = raw.loc[1:, cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    mat.index = sgRNAs
    return mat.astype(np.int64)

# ---- donor = Pre2 (from 'preinfection samples for 24 hpi...') ----
j0 = raw.columns.get_loc("preinfection samples for 24 hpi of murine pneumonia model")
pre_cols = list(raw.columns[j0 : j0 + 2])  # (Pre1, Pre2)
donor_col = next(c for c in pre_cols if str(sample_names[c]) == "Pre2")
donor_counts = counts_for_cols([donor_col]).iloc[:, 0]  # pd.Series, index=sgRNA####

# ---- recipients = 24 hpi mice WITH dox (Mice_plusDox_*) ----
i0 = raw.columns.get_loc("24 hpi of murine pneumonia model")
i1 = raw.columns.get_loc("preinfection samples for 48 hpi of murine pneumonia model")
cols_24hpi = list(raw.columns[i0:i1])

plusdox_cols = [c for c in cols_24hpi if isinstance(sample_names[c], str) and sample_names[c].startswith("Mice_plusDox_")]

recipient_counts_df = counts_for_cols(plusdox_cols).T
recipient_counts_df.index = [sample_names[c] for c in plusdox_cols]   # mice IDs # recipient_counts_df.columns are already normalized sgRNA####

In [7]:
nb = pd.read_csv("Nb_filtered.csv")
nb_dox = nb[(nb["hpi"] == 24) & (nb["condition"] == "Dox")].copy()
nb_dox["sample"] = nb_dox["mouse"].astype(int).map(lambda m: f"Mice_plusDox_{m}")
Nb_by_mouse = dict(zip(nb_dox["sample"], nb_dox["lung"]))

missing = [m for m in recipient_counts_df.index if m not in Nb_by_mouse]
if missing:
    raise ValueError(f"Missing Nb entries for these recipients: {missing}")

Nb_vec = np.array([Nb_by_mouse[m] for m in recipient_counts_df.index], dtype=float)

In [8]:
#run neutrality test
res = neutrality_lrt(donor_counts = donor_counts, recipient_counts = recipient_counts_df, bottleneck = Nb_vec,)

res["feature"] = res["feature"].map(normalize_sgrna_id)
sig = res[res["reject_FDR"]].copy()
sig.to_csv("retest_neutrality_test_Dox_refined.csv", index=False)